# 1.2 — Masked Language Model (ESM / BERT style)

**Mechanism of the day:** stop predicting *forward*. Corrupt a few positions and
predict them from **both** sides — then discover that a model trained this way can
still generate, just not in the way it was built to.

In 1.1 you factorized `p(x)` left-to-right. It gave you an exact likelihood and easy
sampling, but every residue could only see its past. That is a strange constraint
for a protein: residue 12 is influenced by residue 20 every bit as much as by
residue 4 — they may be neighbors in 3D even when they are far apart in sequence.

The masked objective throws out the ordering entirely:

```
corrupt ~15% of positions, then predict them from the ENTIRE rest of the sequence
```

This is the objective behind BERT, and behind **ESM** — the protein language models
everyone uses as feature extractors and zero-shot fitness oracles. What you gain is
bidirectional context. What you give up is bigger than it first looks: **there is no
joint distribution any more.** The model gives you conditionals `p(x_t | x_-t)` and
nothing that multiplies together into `p(x)`.

So this notebook asks two questions and answers both with numbers:
1. What does bidirectional context actually *buy*? (We measure it against 1.1.)
2. If there is no joint, how do you **sample**? (Gibbs — and it fails instructively
   if you do the obvious thing instead.)

We reuse **the exact corpus from 1.1** so every comparison is apples to apples.

**How to use this notebook:** read the concept, implement the reps (they
`raise NotImplementedError`), make the checkpoints (`assert`s) pass. Solutions at the
bottom. Trains in ~25s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (5.6, 3.4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

AA = 'ACDEFGHIKLMNPQRSTVWY'
N_AA = len(AA)
MASK = N_AA                # the [MASK] token -> index 20
VOCAB = N_AA + 1           # 21
STOI = {a: i for i, a in enumerate(AA)}
HYDRO = 'AILMFV'
FREQ = {'A':8.25,'R':5.53,'N':4.06,'D':5.46,'C':1.38,'Q':3.93,'E':6.72,'G':7.07,
        'H':2.27,'I':5.91,'L':9.66,'K':5.80,'M':2.41,'F':3.86,'P':4.74,'S':6.65,
        'T':5.36,'W':1.10,'Y':2.92,'V':6.87}
f = np.array([FREQ[a] for a in AA]); is_h = np.array([a in HYDRO for a in AA])
L, EPS, CORE = 28, 0.05, (0, 3)

# ---- identical corpus to 1.1: amphipathic helices with a hidden random phase ----
p_core = np.where(is_h, f * (1 - EPS) / f[is_h].sum(), f * EPS / f[~is_h].sum())
p_non  = np.where(is_h, f * EPS / f[is_h].sum(),  f * (1 - EPS) / f[~is_h].sum())

def make_corpus(n, L=L):
    phases = rng.integers(0, 7, size=n)
    out = np.zeros((n, L), dtype=np.int64)
    for i in range(L):
        r = (i - phases) % 7
        core = (r == CORE[0]) | (r == CORE[1])
        for m, p in ((core, p_core), (~core, p_non)):
            k = int(m.sum())
            if k:
                out[m, i] = rng.choice(N_AA, size=k, p=p)
    return out, phases

def decode(row):
    return ''.join(AA[i] for i in row)

def pattern(seq):
    return ''.join('H' if c in HYDRO else '.' for c in seq)

def heptad_score(seq):
    '''From 1.1: best-over-phases match to the ideal a/d pattern.'''
    return max(sum(1 for i, c in enumerate(seq) if (((i - phi) % 7) in CORE) == (c in HYDRO))
               / len(seq) for phi in range(7))

train, train_phase = make_corpus(8000)
val, val_phase = make_corpus(1000)

# The floor for a BIDIRECTIONAL model: with 27 residues of context the phase is
# essentially certain, so all that is left is the per-slot emission entropy.
H_known = (2/7) * -(p_core*np.log(p_core)).sum() + (5/7) * -(p_non*np.log(p_non)).sum()
AR_PPL_FROM_1_1 = 13.01     # what your left-to-right model scored in 1.1

print('corpus', train.shape, '| MASK token =', MASK)
print('bidirectional floor: ppl %.2f   (1.1 autoregressive managed %.2f)'
      % (math.exp(H_known), AR_PPL_FROM_1_1))
print()
for r in train[:3]:
    print(decode(r), ' ', pattern(decode(r)))

## Part 1 — the corruption recipe (and why it is weirdly specific)

The obvious objective is "replace 15% of positions with `[MASK]` and predict them."
The real BERT/ESM recipe is fussier. Of the ~15% of positions you select:

- **80%** become `[MASK]`
- **10%** become a **random** amino acid
- **10%** are **left completely unchanged** (but still scored)

That looks arbitrary. It fixes a specific bug. If `[MASK]` were the only corruption,
the model would learn "output a real prediction only where I see `[MASK]`, elsewhere
just copy the input" — a shortcut that is perfect at training time and useless
afterwards, because `[MASK]` **never appears at inference**. That is a train/test
mismatch baked straight into the objective.

The 10% random and 10% unchanged positions destroy the shortcut: the model can no
longer trust the token it is looking at, so it must build a real prediction for
*every* position from the surrounding context.

One honest consequence to keep in mind: on that 10% "unchanged" slice the true
answer is sitting right there in the input, so the model can peek. **Training loss
is therefore optimistic** — it is not the pseudo-perplexity we report later.

### Rep 1 — `mask_batch(data, batch_size, mask_prob=0.15)`
Sample a batch, select ~`mask_prob` of positions, and apply 80/10/10.
Return `(x, y, sel)`: corrupted input, true targets, and a boolean `[B, L]` marking
the **selected** positions (the ones that get scored — including the unchanged 10%).
Unselected positions must be identical in `x` and `y`.

In [ ]:
def mask_batch(data, batch_size, mask_prob=0.15):
    '''BERT 80/10/10 corruption. Returns (x, y, sel).'''
    # YOUR CODE HERE
    # hint: y = the clean batch; x = y.clone()
    #       sel = rng.random((batch_size, L)) < mask_prob        -> which are scored
    #       r   = rng.random((batch_size, L))                    -> which corruption
    #       r < 0.8 -> MASK;  0.8 <= r < 0.9 -> random AA;  r >= 0.9 -> leave alone
    raise NotImplementedError

# --- checkpoint ---
xb, yb, sb = mask_batch(train, 512)
assert xb.shape == (512, L) and yb.shape == (512, L), 'wrong shape'
assert sb.dtype == torch.bool, 'sel must be a boolean mask'
assert (xb[~sb] == yb[~sb]).all(), 'unselected positions must be untouched'
frac_sel = sb.float().mean().item()
frac_changed = (xb != yb).float().mean().item()
frac_mask = (xb == MASK).float().mean().item()
assert 0.10 < frac_sel < 0.20, 'should select ~15% of positions'
assert frac_changed < frac_sel, 'the untouched 10% means fewer changes than selections'
print('selected %.3f | actually changed %.3f | replaced by [MASK] %.3f'
      % (frac_sel, frac_changed, frac_mask))
print('~80%% of selections became [MASK], and ~10%% were left alone on purpose ✓')

### Rep 2 — `masked_loss(logits, y, sel)`
Cross-entropy **only at the selected positions**. The other ~85% produce no gradient
at all — which is why an MLM needs more steps than the autoregressive model of 1.1,
where every position was a training signal.

In [ ]:
def masked_loss(logits, y, sel):
    '''Mean cross-entropy over selected positions only.'''
    # YOUR CODE HERE
    # hint: boolean indexing does all the work: logits[sel] is [n_selected, VOCAB]
    raise NotImplementedError

# --- checkpoint ---
fake = torch.zeros(4, L, VOCAB)
y_ = torch.zeros(4, L, dtype=torch.long)
s_ = torch.zeros(4, L, dtype=torch.bool); s_[0, 0] = True
fake[0, 0, 0] = 10.0                        # confident and correct at the ONE scored spot
assert masked_loss(fake, y_, s_).item() < 0.01, 'should be near zero'
s2 = s_.clone(); s2[1, 1] = True            # add a scored spot where the model is clueless
assert masked_loss(fake, y_, s2).item() > 1.0, 'unscored positions must not be averaged in'
print('masked loss ✓ — gradient flows only where you corrupted')

## Part 2 — bidirectional attention: 1.1 minus one line

Here is the punchline of the whole architecture section: the *only* difference
between the transformer in 1.1 and the one here is that we **delete the causal
mask**. That single `-inf` was the entire autoregressive property; without it,
position `t` attends to every position, future included.

Everything else — heads, blocks, residual stream, LayerNorm — is untouched.

### Rep 3 — `bidirectional_self_attention(q, k, v)`
Same as 1.1's function with the masking removed. `q, k, v` are
`[B, n_head, T, d_head]`; return `(out, att)`.

In [ ]:
def bidirectional_self_attention(q, k, v):
    '''Scaled dot-product attention over the WHOLE sequence. Returns (out, att).'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
q, k, v = (torch.randn(2, 4, L, 16) for _ in range(3))
out, att = bidirectional_self_attention(q, k, v)
assert out.shape == (2, 4, L, 16) and att.shape == (2, 4, L, L)
assert torch.allclose(att.sum(-1), torch.ones(2, 4, L), atol=1e-5), 'rows must sum to 1'
assert (att.triu(1) > 0).any(), 'bidirectional means it MUST look at the future'
# the mirror image of 1.1's causality test: now a future token DOES move earlier outputs
v2 = v.clone(); v2[:, :, -1, :] += 99.0
out2, _ = bidirectional_self_attention(q, k, v2)
assert not torch.allclose(out[:, :, 0, :], out2[:, :, 0, :]), 'position 0 should feel the last token'
print('bidirectional attention ✓ — the future now flows backwards (that is the point)')

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, Cc = x.shape
        q, k, v = self.qkv(x).split(Cc, dim=2)
        split = lambda z: z.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        o, _ = bidirectional_self_attention(split(q), split(k), split(v))
        return self.proj(o.transpose(1, 2).contiguous().view(B, T, Cc))

class Block(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                 nn.Linear(4 * d_model, d_model))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class TinyMLM(nn.Module):
    '''Byte-for-byte the 1.1 model, except attention is not masked.'''
    def __init__(self, vocab=VOCAB, d_model=64, n_head=4, n_layer=2, block=L):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d_model), nn.Embedding(block, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.lnf, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, vocab)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.size(1), device=idx.device))
        for b in self.blocks:
            x = b(x)
        return self.head(self.lnf(x))

model = TinyMLM()
print('parameters:', sum(p.numel() for p in model.parameters()), '(same as 1.1)')

## Part 3 — training

Note the honest evaluation: we score the val set with **pure** masking (every scored
position is a real `[MASK]`, no 10% freebies), so the val curve is comparable to the
pseudo-perplexity we compute next. Watch it settle onto the bidirectional floor.

In [ ]:
def val_masked_loss(model, data, n=500, seed=7):
    '''Honest eval: every scored position is a genuine [MASK].'''
    r = np.random.default_rng(seed)
    y = torch.from_numpy(data[:n].copy()); x = y.clone()
    sel = torch.from_numpy(r.random((n, L)) < 0.15)
    x[sel] = MASK
    model.eval()
    with torch.no_grad():
        v = F.cross_entropy(model(x)[sel], y[sel]).item()
    model.train()
    return v

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
hist, t0 = [], time.time()
for step in range(1, 1001):
    xb, yb, sb = mask_batch(train, 128)
    if sb.sum() == 0:
        continue
    loss = masked_loss(model(xb), yb, sb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0 or step == 1:
        vl = val_masked_loss(model, val)
        hist.append((step, loss.item(), vl))
        if step % 250 == 0 or step == 1:
            print('step %4d  train %.4f  val(pure-mask) %.4f  -> ppl %.2f'
                  % (step, loss.item(), vl, math.exp(vl)))
print('trained in %.1fs on CPU' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots()
ax.plot(h[:, 0], h[:, 1], color=BLUE, lw=2, label='train (has 10% freebies)')
ax.plot(h[:, 0], h[:, 2], color=GREEN, lw=2, label='val (pure mask)')
ax.axhline(H_known, color=INK, ls='--', lw=1.5)
ax.axhline(math.log(AR_PPL_FROM_1_1), color=INK, ls=':', lw=1.5)
lbl = dict(va='center', fontsize=9, color=INK, bbox=dict(fc='white', ec='none', pad=1.5))
ax.set_xlim(-20, 1290)
ax.text(1030, H_known, 'bidirectional floor', **lbl)
ax.text(1030, math.log(AR_PPL_FROM_1_1), '1.1 autoregressive', **lbl)
ax.set_xlabel('step'); ax.set_ylabel('loss (nats / masked token)')
ax.set_title('seeing both sides beats seeing only the past')
ax.grid(alpha=.15); ax.legend(frameon=False, fontsize=9)
plt.show()

## Part 4 — there is no likelihood here (so we use a stand-in)

In 1.1 you multiplied `L` conditionals and got an exact `p(x)`. Try that here and it
falls apart: the model gives you `p(x_t | x_-t)` for each `t`, and there is in general
**no joint distribution whose conditionals are those**. They are learned
independently and need not be mutually consistent.

The standard stand-in is **pseudo-perplexity**: mask each position one at a time,
score the true residue, average.

```
pseudo-PPL = exp( mean over t of  -log p(x_t | x_-t) )
```

Be precise about what this is: a fair measure of *"how well do you predict a
held-out residue given its context"*, and **not** a likelihood. It is not
strictly comparable to 1.1's number — the MLM is answering an easier question,
because it gets to see the right-hand context too. That is exactly the comparison
we want, as long as we say out loud what it is.

Efficient trick: instead of one forward pass per (sequence, position), mask
position `t` across the *whole* dataset at once and loop over the `L` positions.

### Rep 4 — `pseudo_perplexity(model, data, n=300)`
Mask one position at a time over the first `n` sequences, accumulate the negative
log-probability of the true residue, and return `exp(mean)`.

In [ ]:
def pseudo_perplexity(model, data, n=300):
    '''exp of the mean NLL of each residue given ALL the others.'''
    # YOUR CODE HERE
    # hint: for t in range(L): x = clean.clone(); x[:, t] = MASK; forward once;
    #       F.log_softmax(logits[:, t, :], -1) then gather the true tokens
    raise NotImplementedError

# --- checkpoint ---
ppl = pseudo_perplexity(model, val)
floor = math.exp(H_known)
assert ppl < AR_PPL_FROM_1_1, 'bidirectional context should beat 1.1'
assert ppl < 12.6, 'expected ~11.9; if much higher, training or masking is off'
print('pseudo-perplexity  %.2f' % ppl)
print('1.1 autoregressive %.2f   (left context only)' % AR_PPL_FROM_1_1)
print('bidirectional floor %.2f  (phase is certain given 27 neighbours)' % floor)
print('\nIt sits on the floor — within noise of optimal. The ~1.1 ppl it gained over')
print('the AR model is precisely the value of reading the right-hand context. ✓')

## Part 5 — generating from a model that was never meant to generate

We have a scorer, not a sampler. There is no "first token" to start from and no
chain rule to walk. So what do you do?

**The tempting shortcut:** feed a row of all-`[MASK]`, take the predicted
distribution at every position, and sample them all in one pass. It is one forward
pass and gives a full sequence. It is also **badly wrong**, and it is worth
understanding exactly why: each position is sampled from its *marginal*, so the
positions are drawn **independently**. Every one of them separately hedges over all
7 possible phases instead of committing to one. You get a sequence whose residues
individually look plausible and jointly agree on nothing.

**The correct move:** **Gibbs sampling**. Resample positions *one at a time*, each
conditioned on the current value of all the others, and repeat for a few sweeps.
Now every choice sees the choices already made, so the sequence can converge on a
single coherent phase.

Watch the two approaches diverge — this is the most important measurement in the
notebook, and it is the reason rung 1.5 exists.

### Rep 5 — `gibbs_sample(model, n, sweeps=4, temperature=1.0)`
Start from all `[MASK]`. For each sweep, visit the `L` positions **in random order**
and resample each from the model's conditional given everything else. Forbid the
model from emitting `[MASK]` itself. Return `n` decoded strings.

In [ ]:
def oneshot_sample(model, n, temperature=1.0):
    '''GIVEN (the tempting shortcut): fill every position independently, one pass.'''
    x = torch.full((n, L), MASK, dtype=torch.long)
    model.eval()
    with torch.no_grad():
        logits = model(x) / temperature
        logits[:, :, MASK] = float('-inf')
        out = torch.multinomial(F.softmax(logits, -1).reshape(-1, VOCAB), 1).reshape(n, L)
    return [decode(r) for r in out.numpy()]

def gibbs_sample(model, n, sweeps=4, temperature=1.0):
    '''Iteratively resample one position at a time, conditioned on all the rest.'''
    # YOUR CODE HERE
    # hint: x = torch.full((n, L), MASK); for each sweep, for t in torch.randperm(L):
    #       logits = model(x)[:, t, :] / temperature ; logits[:, MASK] = -inf
    #       x[:, t] = torch.multinomial(F.softmax(logits, -1), 1).squeeze(1)
    raise NotImplementedError

# --- checkpoint ---
real = [decode(r) for r in train[:500]]
one = oneshot_sample(model, 500)
gib = gibbs_sample(model, 500, sweeps=4)
hs = lambda S: float(np.mean([heptad_score(s) for s in S]))
assert all(len(s) == L for s in gib) and all(c in AA for s in gib for c in s)
assert hs(one) < 0.80, 'one-shot should be near the random baseline'
assert hs(gib) > 0.90, 'gibbs should recover the register'
print('heptad score   real data %.3f | one-shot %.3f | gibbs(4 sweeps) %.3f'
      % (hs(real), hs(one), hs(gib)))
print()
print('one-shot:', one[0], pattern(one[0]))
print('gibbs   :', gib[0], pattern(gib[0]))
print('\nSame model, same weights. Only the DECODING changed. ✓')

In [ ]:
sweeps = [1, 2, 3, 5, 8]
scores = [hs(gibbs_sample(model, 300, sweeps=s)) for s in sweeps]

fig, ax = plt.subplots()
ax.plot([0] + sweeps, [hs(one)] + scores, '-o', color=BLUE, lw=2, ms=6)
ax.axhline(hs(real), color=GREEN, ls='--', lw=1.5)
ax.text(8, hs(real), 'real data', ha='right', va='bottom', fontsize=9, color=GREEN,
        fontweight='bold')
ax.annotate('one-shot\n(independent)', (0, hs(one)), fontsize=9, color=INK,
            xytext=(10, -4), textcoords='offset points')
ax.set_xlabel('Gibbs sweeps'); ax.set_ylabel('heptad score')
ax.set_title('iterative decoding is what makes an MLM a generator')
ax.grid(alpha=.15)
plt.show()
print('Almost all of the gap closes in ONE sweep — because a sweep in random order')
print('is essentially autoregressive generation in a shuffled order. ✓')

### The caveat nobody mentions

Gibbs sampling is only guaranteed to converge to a joint distribution **when one
exists** that has all of the model's conditionals. For a trained MLM, none is
guaranteed to — the conditionals were fit independently and are typically mutually
inconsistent. So this chain has no proven stationary distribution, and where it
lands can depend on the sweep order and the number of sweeps.

You can see the symptom in your own numbers: Gibbs samples usually score *higher*
on `heptad_score` than the real data does. That is not the model being clever, it
is the sampler drifting toward over-typical, cleaner-than-real sequences.

Hold onto this. Rung **1.5 (discrete / masked diffusion)** is precisely the
principled fix: keep the masking objective, but train it under a *schedule* of
corruption levels so that iterative unmasking becomes a proper generative process
with an actual likelihood bound — rather than a heuristic we discovered by poking
a scorer.

## Part 6 — protein instance: infilling (a.k.a. motif scaffolding)

Here is where the masked model beats the autoregressive one outright. Suppose you
have a motif you must keep — a binding site, a catalytic triad, a fixed hydrophobic
anchor — and you want a model to design everything around it.

For an autoregressive model this is awkward: it only generates left-to-right, so
conditioning on residues that come *after* the gap means resampling and filtering.
For a masked model it is the native operation: pin the residues you know, mark the
rest `[MASK]`, and Gibbs the free positions.

This is **motif scaffolding**, one of the flagship applications of protein design.
Below we pin four hydrophobic anchors on a phase-0 register and let the model build
a helix around them.

### Rep 6 — `infill(model, scaffold, sweeps=3, temperature=1.0)`
`scaffold` is a string of length `L` using `'?'` for positions to fill. Encode the
known residues, mask the `'?'`, and Gibbs-resample **only the free positions**,
leaving the pinned ones untouched.

In [ ]:
def infill(model, scaffold, sweeps=3, temperature=1.0):
    '''Fill the '?' positions of a scaffold, keeping every fixed residue.'''
    # YOUR CODE HERE
    # hint: idx = tensor of [MASK if c=='?' else STOI[c]]; free = indices of '?'
    #       Gibbs over `free` ONLY — never touch a pinned position
    raise NotImplementedError

# --- checkpoint ---
scaf = list('?' * L)
for i in (0, 3, 7, 10):        # hydrophobic anchors on the phase-0 a/d register
    scaf[i] = 'L'
scaf = ''.join(scaf)

outs = [infill(model, scaf, sweeps=3) for _ in range(5)]
assert all(len(o) == L for o in outs)
for o in outs:
    assert all(o[i] == 'L' for i in (0, 3, 7, 10)), 'pinned residues must survive'
mean_hs = float(np.mean([heptad_score(o) for o in outs]))
assert mean_hs > 0.85, 'infills should continue the register the anchors imply'

print('scaffold:', scaf)
for o in outs:
    print('  ', o, pattern(o), 'heptad %.2f' % heptad_score(o))
print('\nmean heptad %.3f — from 4 anchors it inferred the phase and completed' % mean_hs)
print('the register across all 24 free positions. That is motif scaffolding. ✓')

## Reflection — what just transferred

- **The mask was the only thing making 1.1 autoregressive.** Delete one `-inf` and
  the same architecture becomes bidirectional. Objectives, not architectures, decide
  what a model is.
- **Bidirectional context is worth something measurable** — here ~1.1 perplexity,
  and we could say exactly why: two-sided evidence pins the hidden phase. This is
  why ESM embeddings are such good features, and why ESM is a strong zero-shot
  fitness predictor despite never being trained to score fitness.
- **You paid for it with the joint distribution.** No likelihood, only conditionals,
  and pseudo-perplexity is a stand-in that must be quoted honestly.
- **Independent decoding is the classic trap.** Sampling every position at once from
  a bidirectional model gives residues that each hedge over every hypothesis and
  jointly commit to none. `0.69` vs `0.97` on the same weights — the decoder, not
  the model, was the difference.
- **Iterative refinement is the fix**, and once you have seen it here you will
  recognize it everywhere: it *is* diffusion's reverse process, wearing different
  notation.
- **Infilling is native to masked models**, which is why this family underpins
  motif scaffolding and inverse folding (2.4).

**Next rep:** `1.3 Continuous diffusion (DDPM)` — leave sequences behind for a
notebook and build the forward/reverse process from scratch on toy 2D data, where
you can see every step. Then 1.5 brings diffusion *back* to sequences and fixes the
Gibbs hand-waving above.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def mask_batch(data, batch_size, mask_prob=0.15):
    ix = rng.integers(0, len(data), size=batch_size)
    y = torch.from_numpy(data[ix].copy())
    x = y.clone()
    sel = torch.from_numpy(rng.random((batch_size, L)) < mask_prob)
    r = torch.from_numpy(rng.random((batch_size, L)))
    x[sel & (r < 0.8)] = MASK                                   # 80% -> [MASK]
    rand_pos = sel & (r >= 0.8) & (r < 0.9)                     # 10% -> random AA
    x[rand_pos] = torch.from_numpy(rng.integers(0, N_AA, size=int(rand_pos.sum())))
    return x, y, sel                                            # 10% -> left as-is

def masked_loss(logits, y, sel):
    return F.cross_entropy(logits[sel], y[sel])

def bidirectional_self_attention(q, k, v):
    att = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))     # no causal mask
    att = F.softmax(att, dim=-1)
    return att @ v, att

def pseudo_perplexity(model, data, n=300):
    d = torch.from_numpy(data[:n].copy())
    model.eval()
    total = 0.0
    with torch.no_grad():
        for t in range(L):
            x = d.clone(); x[:, t] = MASK
            lp = F.log_softmax(model(x)[:, t, :], dim=-1)
            total += -lp[torch.arange(len(d)), d[:, t]].sum().item()
    return math.exp(total / (len(d) * L))

def gibbs_sample(model, n, sweeps=4, temperature=1.0):
    x = torch.full((n, L), MASK, dtype=torch.long)
    model.eval()
    with torch.no_grad():
        for _ in range(sweeps):
            for t in torch.randperm(L).tolist():
                logits = model(x)[:, t, :] / temperature
                logits[:, MASK] = float('-inf')
                x[:, t] = torch.multinomial(F.softmax(logits, dim=-1), 1).squeeze(1)
    return [decode(row) for row in x.numpy()]

def infill(model, scaffold, sweeps=3, temperature=1.0):
    idx = torch.tensor([[MASK if c == '?' else STOI[c] for c in scaffold]])
    free = [i for i, c in enumerate(scaffold) if c == '?']
    model.eval()
    with torch.no_grad():
        for _ in range(sweeps):
            for t in np.random.permutation(free):
                lg = model(idx)[:, t, :] / temperature
                lg[:, MASK] = float('-inf')
                idx[0, t] = torch.multinomial(F.softmax(lg, dim=-1), 1)
    return decode(idx[0].numpy())

print('reference solutions loaded — re-run the checkpoint cells above')